In [7]:
print("OK")

OK


**To activate medibot ev**\
source ~/anaconda3/etc/profile.d/conda.sh \
conda activate medibot

In [8]:
import os
os.chdir('../')

In [ ]:
%pwd

'd:\\Rushan\\Data Science\\Projects\\RAG\\Medical-Chatbot-with-LLM-RAG-Langchain-Pinecone-Flask-AWS'

## Strucured Chunking

In [9]:
import fitz  # PyMuPDF
import re

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

# -------- CONFIG --------
PDF_PATH = "data\\Medical_book.pdf"
OUTPUT_FILE = "research\\structured_output.txt"
MAX_PAGES = 10  # change for testing

# -------- MAIN LOGIC --------
doc = fitz.open(PDF_PATH)

In [10]:
structured_data = []

state = {
    "term": None,
    "section": None,
    "prev_section": None,
    "in_key_terms": False,
    "section_buffer": [],
    "key_terms_buffer": []
}


def save_section(page_num):
    term = state["term"]
    section = state["section"]

    if not term or not section:
        return

    buffer = (
        state["key_terms_buffer"]
        if state["in_key_terms"]
        else state["section_buffer"]
    )

    content = " ".join(buffer).strip()
    if not content:
        return

    structured_data.append({
        "term": term,
        "section": section,
        "content": content,
        "page": page_num + 1
    })

    # Clear buffer
    if state["in_key_terms"]:
        state["key_terms_buffer"].clear()
    else:
        state["section_buffer"].clear()


for page_num, page in enumerate(doc):

    blocks = page.get_text("dict")["blocks"]

    for block in blocks:
        for line in block.get("lines", []):
            for span in line.get("spans", []):

                text = clean_text(span["text"])
                if not text or text == "GALE ENCYCLOPEDIA OF MEDICINE 2" or text.isdigit():
                    continue

                font_size = span["size"]
                font_name = span["font"]

                # -------- TERM --------
                if font_size == 15:
                    save_section(page_num)
                    state["term"] = text
                    state["section"] = None
                    continue

                # -------- KEY TERMS START --------
                if font_size == 12.5 and text == "KEY TERMS":
                    state["prev_section"] = state["section"]
                    state["section"] = "KEY TERMS"
                    state["in_key_terms"] = True
                    continue

                # -------- SECTION --------
                if font_size == 11 or (text == "Resources" and font_name == "Optima-Bold"):

                    # If exiting KEY TERMS
                    if state["in_key_terms"]:
                        save_section(page_num)
                        state["in_key_terms"] = False
                        state["section"] = state["prev_section"]

                    # Save previous section if exists
                    save_section(page_num)

                    state["prev_section"] = state["section"]
                    state["section"] = text
                    continue

                # -------- KEY TERMS END --------
                if state["in_key_terms"] and font_name == "Times-Roman":
                    save_section(page_num)
                    state["in_key_terms"] = False
                    state["section"] = state["prev_section"]
                    continue

                # -------- CONTENT --------
                if state["term"] and state["section"]:
                    if state["in_key_terms"]:
                        state["key_terms_buffer"].append(text)
                    else:
                        state["section_buffer"].append(text)


# Final flush
save_section(page_num)

In [11]:
structured_data[:5]

[{'term': 'Abdominal ultrasound',
  'section': 'Definition',
  'content': 'Ultrasound technology allows doctors to “see” inside a patient without resorting to surgery. A transmit- ter sends high frequency sound waves into the body, where they bounce off the different tissues and organs to produce a distinctive pattern of echoes. A receiver “hears” the returning echo pattern and forwards it to a computer, which translates the data into an image on a television screen. Because ultrasound can distinguish subtle variations between soft, fluid-filled tissues, it is particularly useful in providing diagnostic images of the abdomen. Ultrasound can also be used in treatment.',
  'page': 15},
 {'term': 'Abdominal ultrasound',
  'section': 'Purpose',
  'content': 'The potential medical applications of ultrasound were first recognized in the 1940s as an outgrowth of the sonar technology developed to detect submarines during World War II. The first useful medical images were pro- duced in the earl

In [8]:
OUTPUT_FILE = "research\\structured_output2.txt"
# -------- SAVE TO TXT --------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in structured_data:
        f.write(f"\n=== TERM: {item['term']} ===\n")
        f.write(f"[Section: {item['section']} | Page: {item['page']}]\n")
        f.write(f"{item['content']}\n")

print(f"✅ Structured output saved to {OUTPUT_FILE}")

✅ Structured output saved to research\structured_output2.txt


In [12]:
len(structured_data)

2534

## Creating langchain documents

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

documents = []

for item in structured_data:
    term = item["term"]
    section = item["section"]
    content = item["content"]
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
    chunks = text_splitter.split_text(content)
    
    for chunk in chunks:
        # Format text with term and section as headers
        formatted_text = f"{term} -> {section} ->\n\n{chunk}"
        
        documents.append(
            Document(
                page_content=formatted_text,
                metadata={"term": term, "section": section, "page": item["page"]},
            )
        )

In [15]:
documents[:10]

[Document(metadata={'term': 'Abdominal ultrasound', 'section': 'Definition', 'page': 15}, page_content='Abdominal ultrasound -> Definition ->\n\nUltrasound technology allows doctors to “see” inside a patient without resorting to surgery. A transmit- ter sends high frequency sound waves into the body, where they bounce off the different tissues and organs to produce a distinctive pattern of echoes. A receiver “hears” the returning echo pattern and forwards it to a computer, which translates the data into an image on a television screen. Because ultrasound can distinguish subtle variations between soft, fluid-filled tissues, it is particularly useful in providing diagnostic images of the abdomen. Ultrasound can also be used in treatment.'),
 Document(metadata={'term': 'Abdominal ultrasound', 'section': 'Purpose', 'page': 16}, page_content='Abdominal ultrasound -> Purpose ->\n\nThe potential medical applications of ultrasound were first recognized in the 1940s as an outgrowth of the sonar

In [16]:
len(documents)

4015

## Embeddings(HF-MiniLM) + VectorStore(Pinecone)

In [1]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\mohdr\AppData\Local\Temp\ipykernel_14328\30168171.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\mohdr\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [26]:
from dotenv import load_dotenv

load_dotenv()

True

In [27]:
from pinecone import Pinecone

pc = Pinecone()

In [30]:
pc

In [ ]:
from pinecone import ServerlessSpec 

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [32]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

In [3]:
from langchain_pinecone import PineconeVectorStore

index_name = "medical-chatbot"

vector_store = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [36]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})

In [37]:
retrieved_docs = retriever.invoke("What is Acne? Give Definition, Causes, Symptoms, and Treatment.")
retrieved_docs

[Document(id='55e7b44a-a356-4111-860f-72fc2866cf3f', metadata={'page': 38.0, 'section': 'Definition', 'term': 'Acne'}, page_content='Acne -> Definition ->\n\nAcne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.'),
 Document(id='70090082-2afd-483b-a8fa-853986463e03', metadata={'page': 38.0, 'section': 'Causes and symptoms', 'term': 'Acne'}, page_content='Acne -> Causes and symptoms ->\n\nThe exact cause of acne is unknown. Several risk factors have been identified: • Age. Due to the hormonal changes they experience, teenagers are more likely to develop acne. • Gender. Boys have more severe acne and develop it more often than girls. • Disease. Hormonal disorders can complicate acne in girls. • Heredity. Individuals with a family history of acne have greater susceptibility to the disease. • Hormonal changes. Acne can flare up before menstrua- tion, during pregnancy 

In [150]:
vector_store.similarity_search("What is Acne? Give Definition, Causes, Symptoms, and Treatment.", k=5)

[Document(id='55e7b44a-a356-4111-860f-72fc2866cf3f', metadata={'page': 38.0, 'section': 'Definition', 'term': 'Acne'}, page_content='Acne -> Definition ->\n\nAcne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.'),
 Document(id='70090082-2afd-483b-a8fa-853986463e03', metadata={'page': 38.0, 'section': 'Causes and symptoms', 'term': 'Acne'}, page_content='Acne -> Causes and symptoms ->\n\nThe exact cause of acne is unknown. Several risk factors have been identified: • Age. Due to the hormonal changes they experience, teenagers are more likely to develop acne. • Gender. Boys have more severe acne and develop it more often than girls. • Disease. Hormonal disorders can complicate acne in girls. • Heredity. Individuals with a family history of acne have greater susceptibility to the disease. • Hormonal changes. Acne can flare up before menstrua- tion, during pregnancy 

### with BM25

In [ ]:
import pickle

with open("research\\docs.pkl", "wb") as f:
    pickle.dump(documents, f)

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

with open("research\\docs.pkl", "rb") as f:
    loaded_docs = pickle.load(f)

loaded_docs

[Document(metadata={'term': 'Abdominal ultrasound', 'section': 'Definition', 'page': 15, 'text': 'Abdominal ultrasound -> Definition ->\n\nUltrasound technology allows doctors to “see” inside a patient without resorting to surgery. A transmit- ter sends high frequency sound waves into the body, where they bounce off the different tissues and organs to produce a distinctive pattern of echoes. A receiver “hears” the returning echo pattern and forwards it to a computer, which translates the data into an image on a television screen. Because ultrasound can distinguish subtle variations between soft, fluid-filled tissues, it is particularly useful in providing diagnostic images of the abdomen. Ultrasound can also be used in treatment.'}, page_content='Abdominal ultrasound -> Definition ->\n\nUltrasound technology allows doctors to “see” inside a patient without resorting to surgery. A transmit- ter sends high frequency sound waves into the body, where they bounce off the different tissues a

In [80]:
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mohdr\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [93]:
from nltk.tokenize import word_tokenize

bm25_retriever = BM25Retriever.from_documents(
    documents=documents,
    preprocess_func=word_tokenize)
bm25_retriever.k = 5

In [94]:
retrieved_docs = bm25_retriever.invoke("What is Acne? Give Definition, Causes, Symptoms, and Treatment.")
retrieved_docs

[Document(metadata={'term': 'Acne', 'section': 'Definition', 'page': 38, 'text': 'Acne -> Definition ->\n\nAcne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.'}, page_content='Acne -> Definition ->\n\nAcne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.'),
 Document(metadata={'term': 'Aspirin', 'section': 'Resources', 'page': 393, 'text': 'Aspirin -> Resources ->\n\nPERIODICALS “How to Give Medicine to Children” (Includes related article on health risks of aspirin for children). FDA Consumer (Jan./Feb. 1996): 6. “The Miracle Drug in Your Medicine Cabinet.” American Health (Jan./Feb. 1996): 67. “No Aspirin, Please.” Current Health (Dec. 1992): 12. “What’s the Best Pain Reliever? Depends on Your Pain.” Con- sumer Reports , May 1996, 62. Nanc

In [123]:
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, retriever], weights=[0.5, 0.5])

retrieved_docs = ensemble_retriever.invoke("What is Acne? Give Definition, Causes, Symptoms, and Treatment.")
retrieved_docs

[Document(metadata={'term': 'Acne', 'section': 'Definition', 'page': 38, 'text': 'Acne -> Definition ->\n\nAcne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.'}, page_content='Acne -> Definition ->\n\nAcne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.'),
 Document(metadata={'term': 'Aspirin', 'section': 'Resources', 'page': 393, 'text': 'Aspirin -> Resources ->\n\nPERIODICALS “How to Give Medicine to Children” (Includes related article on health risks of aspirin for children). FDA Consumer (Jan./Feb. 1996): 6. “The Miracle Drug in Your Medicine Cabinet.” American Health (Jan./Feb. 1996): 67. “No Aspirin, Please.” Current Health (Dec. 1992): 12. “What’s the Best Pain Reliever? Depends on Your Pain.” Con- sumer Reports , May 1996, 62. Nanc

### with LLM structured output

In [78]:
term_set = set([doc.metadata["term"] for doc in documents])
section_set = set([doc.metadata["section"] for doc in documents])

In [79]:
with open("data\\term_set.txt", "w") as term_file:
        for term in term_set:
            term_file.write(f"{term}\n")

with open("data\\section_set.txt", "w") as section_file:
    for section in section_set:
        section_file.write(f"{section}\n")

In [31]:
len(term_set), len(section_set)

(281, 33)

In [5]:
from pydantic import BaseModel
from typing import List, Optional

class QueryMetadata(BaseModel):
    term: Optional[str]
    section: List[str]

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

prompt = ChatPromptTemplate.from_template("""
Extract structured medical query information.

Rules:
- Extract section names ONLY from this list:
  {section_set}
- If not mentioned, return empty list
- Try to extract the medical term, but it can be approximate from this list :
  {term_set}

Query: {query}

Return JSON:
{{
  "term": "...",
  "section": [...]
}}
""")

chain = prompt | llm.with_structured_output(QueryMetadata)

In [148]:
result = chain.invoke({"query": "What we call medicines that relieve pain and how do they work?", "section_set": list(section_set)})

print(result)

term='medicines that relieve pain' section=['Definition', 'Description']


In [ ]:
result = chain.invoke({"query": "Give Definition, Description, Causes, Symptoms, and Treatment for Acne", 
                       "section_set": list(section_set), 
                       "term_set": list(term_set)})

print(result)

term='Acne' section=['Definition', 'Description', 'Causes and symptoms', 'Symptoms', 'Treatment']


In [40]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":8,
                                                                               "filter": {"term": result.term, 
                                                                                          "section": {"$in": result.section}}})

retrieved_docs = retriever.invoke("Give Definition, Description, Causes, Symptoms, and Treatment for Acne")
retrieved_docs

[Document(id='55e7b44a-a356-4111-860f-72fc2866cf3f', metadata={'page': 38.0, 'section': 'Definition', 'term': 'Acne'}, page_content='Acne -> Definition ->\n\nAcne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.'),
 Document(id='6dd7eca6-7601-481c-b7b7-03d2036f1945', metadata={'page': 38.0, 'section': 'Causes and symptoms', 'term': 'Acne'}, page_content='Acne -> Causes and symptoms ->\n\nmake them worse. • Cosmetics. Oil-based makeup and hair sprays worsen acne. • Environment. Exposure to oils and greases, polluted air, and sweating in hot weather aggravate acne. • Stress . Emotional stress may contribute to acne. Acne is usually not conspicuous, although inflamed lesions may cause pain , tenderness, itching , or swelling. The most troubling aspects of these lesions are the nega- tive cosmetic effects and potential for scarring. Some people, especially teenagers, 

In [41]:
result = chain.invoke({"query": "What we call medicines that relieve pain and how do they work?", 
                       "section_set": list(section_set), 
                       "term_set": list(term_set)})

print(result)

term='Analgesics' section=['Definition', 'Description']


In [149]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":8,
                                                                               "filter": {"section": {"$in": result.section}}})

retrieved_docs = retriever.invoke("What we call medicines that relieve pain and how do they work?")
retrieved_docs

[Document(id='3ea8a2f0-5b68-46f8-934a-5571c44a507c', metadata={'page': 187.0, 'section': 'Definition', 'term': 'Analgesics'}, page_content='Analgesics -> Definition ->\n\nAnalgesics are medicines that relieve pain .'),
 Document(id='f9c6c6cc-30b8-4929-8eda-8a784d3994dc', metadata={'page': 189.0, 'section': 'Description', 'term': 'Analgesics, opioid'}, page_content='Analgesics, opioid -> Description ->\n\nOpioid analgesics relieve pain by acting directly on the central nervous system. However, this can also lead to unwanted side effects, such as drowsiness, dizziness , breathing problems, and physical or mental dependence. Among the drugs in this category are codeine, pro- poxyphene (Darvon), propoxyphene and acetaminophen (Darvocet N), meperidine (Demerol), hydromorphone (Dilaudid), morphine, oxycodone, oxycodone and aceta- minophen (Percocet, Roxicet), and hydrocodone and acetaminophen (Lortab, Anexsia). These drugs come in many forms—tablets, syrups, suppositories, and injec- tions, 

In [ ]:
query = "Tell me about a disease that affects memory, its causes, prevention and treatment?"

vector_store.similarity_search(query, k=8)

[Document(id='f861b793-2711-4667-a729-04cb9e421ad0', metadata={'page': 167.0, 'section': 'Treatment', 'term': 'Amnesia'}, page_content='Amnesia -> Treatment ->\n\nTreatment depends on the root cause of amnesia and is handled on an individual basis. Regardless of cause, cognitive rehabilitation may be helpful in learning strategies to cope with memory impairment. Amnesia Amygdala Hippocampus Memory loss may result from bilateral damage to the limbic system of the brain responsible for memory storage, pro- cessing, and recall. (Illustration by Electronic Illustrators Group).'),
 Document(id='db61fd00-e89f-4564-9229-76bac274e5cf', metadata={'page': 151.0, 'section': 'Causes and symptoms', 'term': 'Alzheimer’s disease'}, page_content='Alzheimer’s disease -> Causes and symptoms ->\n\nCauses The cause or causes of Alzheimer’s disease are unknown. Some strong leads have been found through recent research, however, and these have also given some theoretical support to several new experimental 

In [ ]:
result = chain.invoke({"query": query, 
                       "section_set": list(section_set), 
                       "term_set": list(term_set)})

print(result)

term='Alzheimer’s disease' section=['Causes and symptoms', 'Prevention', 'Treatment']


In [57]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":7,
                                                                               "filter": {"section": {"$in": result.section}}})
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='f861b793-2711-4667-a729-04cb9e421ad0', metadata={'page': 167.0, 'section': 'Treatment', 'term': 'Amnesia'}, page_content='Amnesia -> Treatment ->\n\nTreatment depends on the root cause of amnesia and is handled on an individual basis. Regardless of cause, cognitive rehabilitation may be helpful in learning strategies to cope with memory impairment. Amnesia Amygdala Hippocampus Memory loss may result from bilateral damage to the limbic system of the brain responsible for memory storage, pro- cessing, and recall. (Illustration by Electronic Illustrators Group).'),
 Document(id='db61fd00-e89f-4564-9229-76bac274e5cf', metadata={'page': 151.0, 'section': 'Causes and symptoms', 'term': 'Alzheimer’s disease'}, page_content='Alzheimer’s disease -> Causes and symptoms ->\n\nCauses The cause or causes of Alzheimer’s disease are unknown. Some strong leads have been found through recent research, however, and these have also given some theoretical support to several new experimental 

In [53]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5,
                                                                               "filter": {"term": result.term, 
                                                                                            "section": {"$in": result.section}}})

retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='db61fd00-e89f-4564-9229-76bac274e5cf', metadata={'page': 151.0, 'section': 'Causes and symptoms', 'term': 'Alzheimer’s disease'}, page_content='Alzheimer’s disease -> Causes and symptoms ->\n\nCauses The cause or causes of Alzheimer’s disease are unknown. Some strong leads have been found through recent research, however, and these have also given some theoretical support to several new experimental treatments. Alzheimer’s disease At first AD destroys neurons (nerve cells) in parts of the brain that control memory, including the hippocam- pus, which is a structure deep in the deep that controls short-term memory. As these neurons in the hippocampus stop functioning, the short-term memory of the person fails, and the ability to perform familiar tasks decreases. Later AD affects the cerebral cortex, particularly the areas responsible for language and reasoning; this language skills are lost and the ability to make judgments is changed. Personality changes occur, which may i

In [68]:
query = "Tell me about a disease that affects memory, its causes, prevention and treatment?"

# 1. Normal retrieval
docs_semantic = vector_store.similarity_search(query, k=8)

# 2. Section Filtered retrieval
result = chain.invoke({"query": query, "section_set": list(section_set), "term_set": list(term_set)})
docs_sec_filtered = vector_store.similarity_search(
    query,
    k=8,
    filter={"section": {"$in": result.section}}
)

# 3. Term + Section Filtered retrieval
docs_filtered = vector_store.similarity_search(
    query,
    k=8,
    filter={"term": result.term, "section": {"$in": result.section}}
)

# 4. Combine + deduplicate
all_docs = {doc.page_content: doc for doc in docs_semantic + docs_sec_filtered + docs_filtered}
docs = list(all_docs.values())

In [69]:
result

QueryMetadata(term='Alzheimer’s disease', section=['Causes and symptoms', 'Prevention', 'Treatment'])

In [70]:
docs_semantic

[Document(id='f861b793-2711-4667-a729-04cb9e421ad0', metadata={'page': 167.0, 'section': 'Treatment', 'term': 'Amnesia'}, page_content='Amnesia -> Treatment ->\n\nTreatment depends on the root cause of amnesia and is handled on an individual basis. Regardless of cause, cognitive rehabilitation may be helpful in learning strategies to cope with memory impairment. Amnesia Amygdala Hippocampus Memory loss may result from bilateral damage to the limbic system of the brain responsible for memory storage, pro- cessing, and recall. (Illustration by Electronic Illustrators Group).'),
 Document(id='db61fd00-e89f-4564-9229-76bac274e5cf', metadata={'page': 151.0, 'section': 'Causes and symptoms', 'term': 'Alzheimer’s disease'}, page_content='Alzheimer’s disease -> Causes and symptoms ->\n\nCauses The cause or causes of Alzheimer’s disease are unknown. Some strong leads have been found through recent research, however, and these have also given some theoretical support to several new experimental 

In [71]:
docs_sec_filtered

[Document(id='f861b793-2711-4667-a729-04cb9e421ad0', metadata={'page': 167.0, 'section': 'Treatment', 'term': 'Amnesia'}, page_content='Amnesia -> Treatment ->\n\nTreatment depends on the root cause of amnesia and is handled on an individual basis. Regardless of cause, cognitive rehabilitation may be helpful in learning strategies to cope with memory impairment. Amnesia Amygdala Hippocampus Memory loss may result from bilateral damage to the limbic system of the brain responsible for memory storage, pro- cessing, and recall. (Illustration by Electronic Illustrators Group).'),
 Document(id='db61fd00-e89f-4564-9229-76bac274e5cf', metadata={'page': 151.0, 'section': 'Causes and symptoms', 'term': 'Alzheimer’s disease'}, page_content='Alzheimer’s disease -> Causes and symptoms ->\n\nCauses The cause or causes of Alzheimer’s disease are unknown. Some strong leads have been found through recent research, however, and these have also given some theoretical support to several new experimental 

In [72]:
docs_filtered

[Document(id='db61fd00-e89f-4564-9229-76bac274e5cf', metadata={'page': 151.0, 'section': 'Causes and symptoms', 'term': 'Alzheimer’s disease'}, page_content='Alzheimer’s disease -> Causes and symptoms ->\n\nCauses The cause or causes of Alzheimer’s disease are unknown. Some strong leads have been found through recent research, however, and these have also given some theoretical support to several new experimental treatments. Alzheimer’s disease At first AD destroys neurons (nerve cells) in parts of the brain that control memory, including the hippocam- pus, which is a structure deep in the deep that controls short-term memory. As these neurons in the hippocampus stop functioning, the short-term memory of the person fails, and the ability to perform familiar tasks decreases. Later AD affects the cerebral cortex, particularly the areas responsible for language and reasoning; this language skills are lost and the ability to make judgments is changed. Personality changes occur, which may i

In [73]:
len(docs)

15

In [76]:
docs

[Document(id='f861b793-2711-4667-a729-04cb9e421ad0', metadata={'page': 167.0, 'section': 'Treatment', 'term': 'Amnesia'}, page_content='Amnesia -> Treatment ->\n\nTreatment depends on the root cause of amnesia and is handled on an individual basis. Regardless of cause, cognitive rehabilitation may be helpful in learning strategies to cope with memory impairment. Amnesia Amygdala Hippocampus Memory loss may result from bilateral damage to the limbic system of the brain responsible for memory storage, pro- cessing, and recall. (Illustration by Electronic Illustrators Group).'),
 Document(id='db61fd00-e89f-4564-9229-76bac274e5cf', metadata={'page': 151.0, 'section': 'Causes and symptoms', 'term': 'Alzheimer’s disease'}, page_content='Alzheimer’s disease -> Causes and symptoms ->\n\nCauses The cause or causes of Alzheimer’s disease are unknown. Some strong leads have been found through recent research, however, and these have also given some theoretical support to several new experimental 

### + ReRanking ( Cross Encoder )

In [74]:
from sentence_transformers import CrossEncoder

# Load model
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Prepare pairs
pairs = [(query, doc.page_content) for doc in docs]

# Get scores
scores = reranker.predict(pairs)

# Sort docs
ranked_docs = [doc for _, doc in sorted(zip(scores, docs), reverse=True)][:8]

In [75]:
scores

array([ 0.22636929,  0.5366932 , -0.4337074 , -5.7953734 , -3.0848672 ,
       -0.29234186, -3.031935  , -1.0499127 , -2.8538861 , -6.752926  ,
       -6.397528  , -5.6726713 , -3.347662  , -5.5157294 , -7.4701185 ],
      dtype=float32)

In [7]:
ranked_docs

NameError: name 'ranked_docs' is not defined

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    base_url="https://openrouter.ai/api/v1"
)

system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use five sentences maximum and keep the "
    "answer concise. Mention page numbers from the book along with the answer."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

chain = prompt | model | StrOutputParser()

query = "What is Acne? Give Definition, Causes, Symptoms, and Treatment."
context = "\n\n---\n\n".join([f'Page Number- {doc.metadata["page"]}: {doc.page_content}' for doc in ranked_docs])

response = chain.invoke({"input": query, "context": context})

print("Response:\n", response)

Response:
 **Acne** (also called acne vulgaris) is a common skin disorder in which the pores of the skin become clogged with oil, dead skin cells, and bacteria. It typically presents as pimples on the face, chest, and back.  

---

### 1. Definition  
Acne is a skin disease characterized by pimples that occur when the pores are clogged with oil, dead skin cells, and bacteria.  *(Page 38.0)*  

---

### 2. Causes (risk factors)  
| Cause / Risk Factor | Description |
|---------------------|-------------|
| **Hormonal changes** | Puberty, menstruation, pregnancy, menopause increase androgen levels, leading to excess sebum production. *(Page 38.0)* |
| **Age & Hormones** | Teenagers are most prone; hormonal shifts drive sebum over‑production. *(Page 38.0)* |
| **Gender** | Boys tend to develop more severe acne than girls. *(Page 38.0)* |
| **Genetics** | Family history increases susceptibility. *(Page 38.0)* |
| **Certain medications** | Tranquilizers, antidepressants, antibiotics, oral c